In [0]:
%pip install optuna
%pip install xgboost
%pip install lightgbm
%pip install catboost

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import logging
logging.getLogger("mlflow.tracking.context.registry").setLevel(logging.ERROR)

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tempfile
 
import optuna
import mlflow
import mlflow.xgboost
import mlflow.lightgbm
import mlflow.catboost
from mlflow.models import infer_signature
 
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
 
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.ensemble import RandomForestRegressor

In [0]:
df = spark.read.table("dev_platform.mlops.supplier_delay_features")
target_df = spark.read.table("dev_platform.mlops.ground_truth_total_cost")


In [0]:
merged_df = df.join(
    target_df,
    on="PONUMBER",
    how="inner"
).withColumnRenamed("ground_truth_total_cost", "total_cost")
merged_df = merged_df.toPandas()

if len(merged_df) == 0:
    raise ValueError(
        "merged_df is empty after joining sample_data_prepared with ground_truth_total_cost on "
        "PONUMBER — check that both tables "
        "cover the same data range and that join keys match (casing, dtypes, missing target rows)."
    )

In [0]:
TARGET_COL = "total_cost"
CAT_COLS = []
DROP_COLS = ["PONUMBER", TARGET_COL]
FEATURE_COLS = [c for c in merged_df.columns if c not in DROP_COLS]
 
METRIC_NAMES = ["rmse", "mae", "mape", "wmape", "smape", "r2"]
PRIMARY_METRIC = "rmse"          # metric Optuna optimizes; try "wmape" for a volume-weighted view
N_TRIALS = 20
 
MLFLOW_EXPERIMENT = (
    "/Workspace/Users/danduprolu.stuthi@latentview.com/supplier_delay/"
)

In [0]:
def compute_regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    residual = y_true - y_pred
 
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2 = float(r2_score(y_true, y_pred))
 
    nonzero = y_true != 0
    mape = float(np.mean(np.abs(residual[nonzero] / y_true[nonzero])) * 100) if nonzero.any() else np.nan
 
    abs_actual_sum = np.sum(np.abs(y_true))
    wmape = float(np.sum(np.abs(residual)) / abs_actual_sum * 100) if abs_actual_sum > 0 else np.nan
 
    denom = np.abs(y_true) + np.abs(y_pred)
    smape_mask = denom != 0
    smape = float(np.mean(2 * np.abs(residual[smape_mask]) / denom[smape_mask]) * 100) if smape_mask.any() else np.nan
 
    return {"rmse": rmse, "mae": mae, "mape": mape, "wmape": wmape, "smape": smape, "r2": r2}

In [0]:
def generate_time_splits(dates, n_splits=2, val_weeks=8, step_weeks=4):
    """Builds `n_splits` rolling train/validation windows ending progressively
    closer to the max date. The most recent split gets the highest weight."""
    max_date = pd.to_datetime(dates.max())
    min_date = pd.to_datetime(dates.min())
 
    weights = np.linspace(0.3, 1.0, n_splits)
    weights = weights / weights.sum()
 
    splits = []
    for i in range(n_splits):
        offset_weeks = (n_splits - 1 - i) * step_weeks
        test_end = max_date - pd.Timedelta(weeks=offset_weeks) + pd.Timedelta(days=1)
        test_start = test_end - pd.Timedelta(weeks=val_weeks)
        train_end = test_start
        splits.append({
            "name": f"val_{test_start.date()}_{test_end.date()}",
            "train_start": min_date,
            "train_end": train_end,
            "test_start": test_start,
            "test_end": test_end,
            "weight": float(weights[i]),
        })
    return splits

In [0]:
def build_split(split_cfg, df, feature_cols, target_col):
    train_mask = (df["week"] >= split_cfg["train_start"]) & (df["week"] < split_cfg["train_end"])
    test_mask = (df["week"] >= split_cfg["test_start"]) & (df["week"] < split_cfg["test_end"])

    X_train = df.loc[train_mask, feature_cols].reset_index(drop=True)
    y_train = df.loc[train_mask, target_col].reset_index(drop=True)
    X_test = df.loc[test_mask, feature_cols].reset_index(drop=True)
    y_test = df.loc[test_mask, target_col].reset_index(drop=True)

    if len(y_train) == 0 or len(y_test) == 0:
        raise ValueError(
            f"Split '{split_cfg['name']}' has an empty train or test set "
            f"(n_train={len(y_train)}, n_test={len(y_test)}). "
            f"train=[{split_cfg['train_start'].date()}, {split_cfg['train_end'].date()}), "
            f"test=[{split_cfg['test_start'].date()}, {split_cfg['test_end'].date()}) "
            f"but data spans [{df['week'].min()}, {df['week'].max()}]. "
            "The configured val_weeks/step_weeks/n_splits window falls outside the "
            "available data range — reduce n_splits/val_weeks/step_weeks or check for a data gap."
        )

    return X_train, y_train, X_test, y_test

In [0]:
def get_model_data(model_name, X_train, X_test, cat_cols):
    """All three models here support categorical features natively, so no
    one-hot encoding is needed — just make sure each model sees categoricals
    in the dtype/form it expects."""
    X_train = X_train.copy()
    X_test = X_test.copy()
 
    if model_name == "catboost":
        for col in cat_cols:
            X_train[col] = X_train[col].astype(str)
            X_test[col] = X_test[col].astype(str)
        return X_train, X_test
    
    for col in cat_cols:
        categories = pd.Index(pd.concat([X_train[col], X_test[col]]).astype(str).unique())
        X_train[col] = pd.Categorical(X_train[col].astype(str), categories=categories)
        X_test[col] = pd.Categorical(X_test[col].astype(str), categories=categories)
    return X_train, X_test

In [0]:
HYPERPARAMETER_SPACE = {
    "xgboost": {
        # Tree Structure & Depth
        "n_estimators": ("int", 200, 1500),
        "max_depth": ("int", 3, 10),
        "min_child_weight": ("int", 1, 20),
        # Learning Rate & Regularization
        "learning_rate": ("float", 1e-3, 0.2, True),  # Log scale search
        "gamma": ("float", 0.0, 5.0),
        "reg_alpha": ("float", 1e-8, 10.0, True),     # L1 Regularization
        "reg_lambda": ("float", 1e-8, 10.0, True),    # L2 Regularization
        # Subsampling for Generalization
        "subsample": ("float", 0.5, 0.95),
        "colsample_bytree": ("float", 0.5, 0.95),
        "colsample_bylevel": ("float", 0.6, 1.0),
        # Fixed Parameters
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "random_state": 42,
        "n_jobs": -1,
    },
    "lightgbm": {
        # Tree Structure (num_leaves should generally be < 2^max_depth)
        "n_estimators": ("int", 200, 1500),
        "max_depth": ("int", 3, 12),
        "num_leaves": ("int", 15, 255),
        "min_child_samples": ("int", 10, 100),
        # Learning Rate & Regularization
        "learning_rate": ("float", 1e-3, 0.2, True),
        "reg_alpha": ("float", 1e-8, 10.0, True),
        "reg_lambda": ("float", 1e-8, 10.0, True),
        # Subsampling (Bagging / Feature Fraction)
        "subsample": ("float", 0.5, 0.95),
        "subsample_freq": ("int", 1, 7),
        "colsample_bytree": ("float", 0.5, 0.95),
        # Fixed Parameters
        "objective": "regression",
        "metric": "rmse",
        "random_state": 42,
        "n_jobs": -1,
        "verbosity": -1,
    },
    "catboost": {
        # Tree Structure & Iterations
        "iterations": ("int", 300, 1500),
        "depth": ("int", 4, 10),
        # Learning & Regularization
        "learning_rate": ("float", 1e-3, 0.2, True),
        "l2_leaf_reg": ("float", 1.0, 20.0, True),
        "random_strength": ("float", 1e-3, 10.0, True),
        "bagging_temperature": ("float", 0.0, 1.0),
        # Fixed Parameters
        "loss_function": "RMSE",
        "eval_metric": "RMSE",
        "random_state": 42,
        "verbose": False,
        "thread_count": -1,
    },
    "random_forest": {
        # Tree Ensemble Settings
        "n_estimators": ("int", 100, 500),
        "max_depth": ("int", 5, 25),
        "min_samples_split": ("int", 2, 20),
        "min_samples_leaf": ("int", 1, 10),
        "max_features": ("float", 0.4, 1.0),
        # Fixed Parameters
        "random_state": 42,
        "n_jobs": -1,
    },
}

MODEL_REGISTRY = {
    "xgboost": XGBRegressor,
    "lightgbm": LGBMRegressor,
    "catboost": CatBoostRegressor,
    "random_forest": RandomForestRegressor,
}

In [0]:
def get_trial_params(trial, model_name):
    config = HYPERPARAMETER_SPACE[model_name]
    params = {}
    for name, value in config.items():
        if isinstance(value, tuple):
            param_type = value[0]
            if param_type == "int":
                params[name] = trial.suggest_int(name, value[1], value[2])
            elif param_type == "float":
                log = False if len(value) == 3 else value[3]
                params[name] = trial.suggest_float(name, value[1], value[2], log=log)
            elif param_type == "categorical":
                params[name] = trial.suggest_categorical(name, value[1])
        else:
            params[name] = value
    return params

In [0]:
def merge_fixed_params(model_name, tuned_params):
    """study.best_params only contains the tuned (suggested) params — merge back
    in the fixed/marker entries from HYPERPARAMETER_SPACE for a full param set."""
    config = HYPERPARAMETER_SPACE[model_name]
    merged = dict(tuned_params)
    for name, value in config.items():
        if not isinstance(value, tuple):
            merged[name] = value
    return merged

In [0]:
def build_model(model_name, params, cat_cols):
    params = dict(params)  # never mutate the caller's dict
    if model_name == "catboost":
        params["cat_features"] = cat_cols
    return MODEL_REGISTRY[model_name](**params)

In [0]:
def evaluate_split(params, model_name, entry, cat_cols):
    model = build_model(model_name, params, cat_cols)
    model.fit(entry["X_train_model"], entry["y_train"])
    y_pred = model.predict(entry["X_test_model"])
    metrics = compute_regression_metrics(entry["y_test"], y_pred)
 
    row = {
        "split_name": entry["cfg"]["name"],
        "train_start": str(entry["cfg"]["train_start"]),
        "train_end": str(entry["cfg"]["train_end"]),
        "test_start": str(entry["cfg"]["test_start"]),
        "test_end": str(entry["cfg"]["test_end"]),
        "weight": entry["cfg"]["weight"],
        "n_train": len(entry["y_train"]),
        "n_test": len(entry["y_test"]),
        **metrics,
    }
    return row, model, y_pred

In [0]:
def evaluate_across_splits(params, model_name, split_data, cat_cols, primary_metric="rmse"):
    rows, models, preds = [], [], []
    for entry in split_data:
        row, model, y_pred = evaluate_split(params, model_name, entry, cat_cols)
        rows.append(row)
        models.append(model)
        preds.append(y_pred)
 
    weights = np.array([r["weight"] for r in rows], dtype=float)
    scores = np.array([r[primary_metric] for r in rows], dtype=float)
    weighted_primary = float(np.average(scores, weights=weights))
 
    return rows, weighted_primary, models[-1], split_data[-1]["X_test_model"], split_data[-1]["y_test"], preds[-1]

In [0]:
def log_feature_importance(model, model_name, feature_cols, artifact_name):
    if model_name == "catboost":
        importances = model.get_feature_importance()
    else:
        importances = model.feature_importances_
 
    fi_df = pd.DataFrame({"feature": feature_cols, "importance": importances}).sort_values(
        "importance", ascending=False
    )
    fi_path = f"{tempfile.gettempdir()}/{artifact_name}_feature_importance.csv"
    fi_df.to_csv(fi_path, index=False)
    mlflow.log_artifact(fi_path, artifact_path="feature_importance")
 
    top = fi_df.head(30)
    fig, ax = plt.subplots(figsize=(8, max(4, len(top) * 0.3)))
    ax.barh(top["feature"][::-1], top["importance"][::-1])
    ax.set_title(f"Feature Importance — {artifact_name}")
    ax.set_xlabel("Importance")
    mlflow.log_figure(fig, f"feature_importance/{artifact_name}.png")
    plt.close(fig)
    return fi_df

In [0]:
def log_predictions(y_true, y_pred, artifact_name, index_df=None):
    pred_df = pd.DataFrame({
        "y_true": np.asarray(y_true, dtype=float),
        "y_pred": np.asarray(y_pred, dtype=float),
    })
    pred_df["residual"] = pred_df["y_true"] - pred_df["y_pred"]
    if index_df is not None:
        pred_df = pd.concat([index_df.reset_index(drop=True), pred_df], axis=1)
 
    pred_path = f"{tempfile.gettempdir()}/{artifact_name}_predictions.csv"
    pred_df.to_csv(pred_path, index=False)
    mlflow.log_artifact(pred_path, artifact_path="predictions")

In [0]:
def objective(trial, model_name, split_data, cat_cols, feature_cols, primary_metric=PRIMARY_METRIC):
    raw_params = get_trial_params(trial, model_name)
    rows = []
 
    with mlflow.start_run(run_name=f"trial_{trial.number}", nested=True):
        mlflow.log_param("model_name", model_name)
        mlflow.log_params(raw_params)
        mlflow.log_dict({"features": feature_cols}, "features.json")
 
        for step, entry in enumerate(split_data):
            row, _, _ = evaluate_split(raw_params, model_name, entry, cat_cols)
            rows.append(row)
            for metric_name in METRIC_NAMES:
                mlflow.log_metric(f"{metric_name}_{row['split_name']}", row[metric_name])
            trial.report(row[primary_metric], step=step)
            if trial.should_prune():
                mlflow.set_tag("trial_status", "pruned")
                raise optuna.TrialPruned()
 
        weights = np.array([r["weight"] for r in rows], dtype=float)
        scores = np.array([r[primary_metric] for r in rows], dtype=float)
        weighted_primary = float(np.average(scores, weights=weights))
        mlflow.log_metric(f"weighted_mean_{primary_metric}", weighted_primary)
 
        csv_path = f"{tempfile.gettempdir()}/{model_name}_trial_{trial.number}_split_metrics.csv"
        pd.DataFrame(rows).to_csv(csv_path, index=False)
        mlflow.log_artifact(csv_path, artifact_path="split_metrics")
 
        mlflow.set_tag("trial_status", "completed")
 
    return weighted_primary

In [0]:
mlflow.set_experiment(MLFLOW_EXPERIMENT)
best_params_per_model = {}
weekly = merged_df.copy()
weekly["MINORDER"] = pd.to_numeric(weekly["MINORDER"], errors="coerce")
weekly[TARGET_COL] = pd.to_numeric(weekly[TARGET_COL], errors="coerce")
weekly["week"] = pd.to_datetime(
    weekly[["REQ_YEAR", "REQ_MONTH", "REQ_DAY"]].rename(
        columns={"REQ_YEAR": "year", "REQ_MONTH": "month", "REQ_DAY": "day"}
    )
)

# ---- Split window selection -------------------------------------------------
# Drop sentinel / implausible request dates (1900-01-01 placeholders from missing
# REQ_* fields). They are not real weeks and would otherwise sit in every train set.
MIN_VALID_WEEK = pd.Timestamp("2000-01-01")
MIN_TEST_ROWS = 50
N_SPLITS, VAL_WEEKS, STEP_WEEKS = 2, 8, 4

n_before = len(weekly)
weekly = weekly.loc[weekly["week"] >= MIN_VALID_WEEK].reset_index(drop=True)
print(f"dropped {n_before - len(weekly)} rows with sentinel/implausible week dates")


def pick_split_anchor(dates, n_splits, val_weeks, step_weeks, min_test_rows):
    """The tail of this dataset is very sparse (only a handful of rows per month in
    the most recent year), so anchoring the rolling windows on dates.max() produces
    3-5 row validation sets and meaningless metrics. Walk back through the observed
    weeks and return the most recent anchor where every validation window holds at
    least `min_test_rows` rows."""
    dates = pd.to_datetime(pd.Series(dates)).sort_values()
    for anchor in dates.drop_duplicates().iloc[::-1]:
        counts = []
        for i in range(n_splits):
            test_end = (
                anchor
                - pd.Timedelta(weeks=(n_splits - 1 - i) * step_weeks)
                + pd.Timedelta(days=1)
            )
            test_start = test_end - pd.Timedelta(weeks=val_weeks)
            counts.append(int(((dates >= test_start) & (dates < test_end)).sum()))
        if min(counts) >= min_test_rows:
            return anchor, counts
    raise ValueError(
        f"No anchor date yields >= {min_test_rows} rows in every validation window "
        f"(n_splits={n_splits}, val_weeks={val_weeks}, step_weeks={step_weeks}). "
        "Widen val_weeks or lower MIN_TEST_ROWS."
    )


SPLIT_ANCHOR, anchor_counts = pick_split_anchor(
    weekly["week"], N_SPLITS, VAL_WEEKS, STEP_WEEKS, MIN_TEST_ROWS
)
print(f"split anchor={SPLIT_ANCHOR.date()} (validation row counts {anchor_counts})")

# generate_time_splits anchors on the max of whatever series it is handed, so clip
# the series to the chosen anchor instead of the sparse absolute max date.
split_dates = weekly.loc[weekly["week"] <= SPLIT_ANCHOR, "week"]

for model_name in MODEL_REGISTRY:
    with mlflow.start_run(run_name=model_name):
        mlflow.log_param("model_name", model_name)
        mlflow.set_tag("model_name", model_name)
        mlflow.set_tag("stage", "hyperparameter_search")
        mlflow.log_dict({"features": FEATURE_COLS}, "features.json")
 
        feature_list_path = f"{tempfile.gettempdir()}/{model_name}_feature_list.csv"
        pd.DataFrame({"feature": FEATURE_COLS}).to_csv(feature_list_path, index=False)
        mlflow.log_artifact(feature_list_path, artifact_path="metadata")
 
        time_splits = generate_time_splits(
            split_dates, n_splits=N_SPLITS, val_weeks=VAL_WEEKS, step_weeks=STEP_WEEKS
        )
 
        split_data = []
        for split_cfg in time_splits:
            X_train, y_train, X_test, y_test = build_split(split_cfg, weekly, FEATURE_COLS, TARGET_COL)
            if len(y_test) < MIN_TEST_ROWS:
                raise ValueError(
                    f"Split '{split_cfg['name']}' has only {len(y_test)} test rows "
                    f"(minimum {MIN_TEST_ROWS}) — metrics computed on this window "
                    "would not be meaningful. Check the split anchor selection."
                )
            X_train_model, X_test_model = get_model_data(model_name, X_train, X_test, CAT_COLS)
            split_data.append({
                "cfg": split_cfg,
                "X_train_model": X_train_model, "y_train": y_train,
                "X_test_model": X_test_model, "y_test": y_test,
            })
            print(
                f"[{model_name}/{split_cfg['name']}] "
                f"train n={len(y_train)}, test n={len(y_test)}"
            )
        mlflow.xgboost.autolog(disable=True)
        mlflow.sklearn.autolog(disable=True)
        study = optuna.create_study(
            direction="minimize",
            study_name=f"{model_name}_study",
            pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=0, interval_steps=1),
        )
        study.optimize(
            lambda trial: objective(trial, model_name, split_data, CAT_COLS, FEATURE_COLS, PRIMARY_METRIC),
            n_trials=N_TRIALS,
        )
 
        # ---- Refit the winning combination exactly once, log it here ----
        best_full_params = merge_fixed_params(model_name, study.best_params)
        best_rows, best_weighted, best_model, best_X_test, best_y_test, best_y_pred = evaluate_across_splits(
            best_full_params, model_name, split_data, CAT_COLS, PRIMARY_METRIC
        )
        best_params_per_model[model_name] = study.best_params
 
        mlflow.log_params({f"best_{k}": v for k, v in study.best_params.items()})
        for row in best_rows:
            for metric_name in METRIC_NAMES:
                mlflow.log_metric(f"best_{metric_name}_{row['split_name']}", row[metric_name])
        mlflow.log_metric(f"best_weighted_mean_{PRIMARY_METRIC}", best_weighted)
 
        best_csv_path = f"{tempfile.gettempdir()}/{model_name}_best_split_metrics.csv"
        pd.DataFrame(best_rows).to_csv(best_csv_path, index=False)
        mlflow.log_artifact(best_csv_path, artifact_path="split_metrics")
 
        # Feature importance + predictions, evaluated on the final (most recent /
        # highest-weighted) split — same convention as the split_data ordering.
        log_feature_importance(best_model, model_name, FEATURE_COLS, artifact_name=f"{model_name}_best")
 
        final_split_cfg = time_splits[-1]
        id_cols_df = weekly.loc[
            (weekly["week"] >= final_split_cfg["test_start"]) & (weekly["week"] < final_split_cfg["test_end"]),
            ["PONUMBER"],
        ].reset_index(drop=True)
        log_predictions(best_y_test, best_y_pred, artifact_name=f"{model_name}_best", index_df=id_cols_df)
 
        # ---- Datasets + metadata (train/validation used for the final split) ----
        final_entry = split_data[-1]
        train_dataset = mlflow.data.from_pandas(
            pd.concat([final_entry["X_train_model"], final_entry["y_train"].rename(TARGET_COL)], axis=1),
            name=f"{model_name}_train_{final_split_cfg['name']}",
        )
        val_dataset = mlflow.data.from_pandas(
            pd.concat([final_entry["X_test_model"], final_entry["y_test"].rename(TARGET_COL)], axis=1),
            name=f"{model_name}_val_{final_split_cfg['name']}",
        )
        mlflow.log_input(train_dataset, context="training")
        mlflow.log_input(val_dataset, context="validation")
 
        dataset_metadata = {
            "n_train": int(len(final_entry["y_train"])),
            "n_val": int(len(final_entry["y_test"])),
            "train_start": str(final_split_cfg["train_start"]),
            "train_end": str(final_split_cfg["train_end"]),
            "test_start": str(final_split_cfg["test_start"]),
            "test_end": str(final_split_cfg["test_end"]),
            "n_features": len(FEATURE_COLS),
            "categorical_features": CAT_COLS,
            "target": TARGET_COL,
        }
        mlflow.log_dict(dataset_metadata, "dataset_metadata.json")
 
        # ---- Model ----
        signature = infer_signature(best_X_test, best_model.predict(best_X_test))
        log_model_fn = {
            "xgboost": mlflow.xgboost.log_model,
            "lightgbm": mlflow.lightgbm.log_model,
            "catboost": mlflow.catboost.log_model,
            "random_forest": mlflow.sklearn.log_model,
        }[model_name]
        log_model_fn(best_model, name="model", signature=signature)  # only log_model call
 
        # ---- Tags ----
        mlflow.set_tag("best_trial_number", study.best_trial.number)
        mlflow.set_tag("n_trials", N_TRIALS)
        mlflow.set_tag("primary_metric", PRIMARY_METRIC)
        mlflow.set_tag("stage", "best_model_refit")
 
        print(f"[{model_name}] best weighted mean {PRIMARY_METRIC}: {best_weighted:.4f}")
        print(f"[{model_name}] best params: {study.best_params}")

dropped 70 rows with sentinel/implausible week dates
split anchor=2025-04-03 (validation row counts [53, 54])
[xgboost/val_2025-01-10_2025-03-07] train n=10608, test n=53


[I 2026-10-07 09:27:12,152] A new study created in memory with name: xgboost_study


[xgboost/val_2025-02-07_2025-04-04] train n=10626, test n=54


[I 2026-10-07 09:27:36,546] Trial 0 finished with value: 20837.57519726073 and parameters: {'n_estimators': 256, 'max_depth': 7, 'min_child_weight': 1, 'learning_rate': 0.02228931613293055, 'gamma': 0.38300082320849804, 'reg_alpha': 1.1837650115232932, 'reg_lambda': 1.0560647120594885e-07, 'subsample': 0.5236274551126514, 'colsample_bytree': 0.8804610640655435, 'colsample_bylevel': 0.688841731847736}. Best is trial 0 with value: 20837.57519726073.
[I 2026-10-07 09:27:54,022] Trial 1 finished with value: 21512.34722522034 and parameters: {'n_estimators': 528, 'max_depth': 9, 'min_child_weight': 19, 'learning_rate': 0.014140148762811297, 'gamma': 4.554087829284959, 'reg_alpha': 0.0006804566264814707, 'reg_lambda': 0.4962463585853441, 'subsample': 0.5643628318396081, 'colsample_bytree': 0.9286853255912831, 'colsample_bylevel': 0.8510858971996962}. Best is trial 0 with value: 20837.57519726073.
[I 2026-10-07 09:28:25,814] Trial 2 finished with value: 29563.068525333685 and parameters: {'n_

[xgboost] best weighted mean rmse: 17540.0153
[xgboost] best params: {'n_estimators': 471, 'max_depth': 3, 'min_child_weight': 6, 'learning_rate': 0.002940671848305232, 'gamma': 3.1399169938586273, 'reg_alpha': 1.33882627274458e-07, 'reg_lambda': 1.9921252773897407e-06, 'subsample': 0.7526912043202398, 'colsample_bytree': 0.5946496568589379, 'colsample_bylevel': 0.7831021329659646}
[lightgbm/val_2025-01-10_2025-03-07] train n=10608, test n=53
[lightgbm/val_2025-02-07_2025-04-04] train n=10626, test n=54


...

[catboost] best weighted mean rmse: 17240.2794
[catboost] best params: {'iterations': 857, 'depth': 4, 'learning_rate': 0.002702017019831149, 'l2_leaf_reg': 1.0964876447737395, 'random_strength': 8.381226542374668, 'bagging_temperature': 0.07042565935558698}
[random_forest/val_2025-01-10_2025-03-07] train n=10608, test n=53


[I 2026-10-07 09:44:52,985] A new study created in memory with name: random_forest_study


[random_forest/val_2025-02-07_2025-04-04] train n=10626, test n=54


[I 2026-10-07 09:45:06,436] Trial 0 finished with value: 21208.407310587023 and parameters: {'n_estimators': 260, 'max_depth': 15, 'min_samples_split': 18, 'min_samples_leaf': 3, 'max_features': 0.839542701006079}. Best is trial 0 with value: 21208.407310587023.
[I 2026-10-07 09:45:19,556] Trial 1 finished with value: 20313.464345141972 and parameters: {'n_estimators': 494, 'max_depth': 9, 'min_samples_split': 15, 'min_samples_leaf': 9, 'max_features': 0.6457347938087594}. Best is trial 1 with value: 20313.464345141972.
[I 2026-10-07 09:45:34,089] Trial 2 finished with value: 19904.636353200145 and parameters: {'n_estimators': 464, 'max_depth': 23, 'min_samples_split': 12, 'min_samples_leaf': 5, 'max_features': 0.4617535204193563}. Best is trial 2 with value: 19904.636353200145.
[I 2026-10-07 09:45:37,441] Trial 3 finished with value: 18298.537780469058 and parameters: {'n_estimators': 134, 'max_depth': 5, 'min_samples_split': 13, 'min_samples_leaf': 6, 'max_features': 0.46091143299675

[random_forest] best weighted mean rmse: 18224.1105
[random_forest] best params: {'n_estimators': 170, 'max_depth': 5, 'min_samples_split': 13, 'min_samples_leaf': 8, 'max_features': 0.4057198279934079}
